# RestaurantRisk Intelligence Platform
## Global Baseline ML Model Training

This notebook downloads historical open inspection data, engineers universal portability features, and evaluates Scikit-Learn / XGBoost models to predict High-Risk failure probability.

In [ ]:
!pip install pandas numpy scikit-learn xgboost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, f1_score
import pickle

### 1. Download Open Data (City of Chicago)
Fetches live public inspection failure data to train the model.

In [ ]:
def load_city_data():
    # URL for Chicago Food Inspections (Socrata API)
    # We'll pull 10,000 recent records for training
    url = "https://data.cityofchicago.org/resource/4ijn-s7e5.json?$limit=10000&$order=inspection_id DESC"
    df = pd.read_json(url)
    return df

print("Downloading dataset...")
raw_df = load_city_data()
print(f"Loaded {len(raw_df)} records.")
raw_df.head(3)

### 2. Feature Engineering (Universal Portability)
To ensure this model works for ANY city/county (not just Chicago), we only extract universal behavioral attributes:
1. **Facility Type Baseline** (Restaurant, Grocery, Mobile)
2. **Risk Category** (Risk 1, Risk 2, Risk 3)
3. **Violations History** (Count of prior failures for this license)
4. **Temporal Decay** (Days since last inspection)

In [ ]:
def engineer_features(df):
    # Target Variable: 1 if Fail, 0 if Pass or Pass w/ Conditions
    df['target_failure'] = df['results'].apply(lambda x: 1 if x == 'Fail' else 0)
    
    # Standardize facility types
    df['facility_type'] = df['facility_type'].str.lower().fillna('unknown')
    # Group into our core 3 categories to match the API
    df['is_restaurant'] = df['facility_type'].apply(lambda x: 1 if 'restaurant' in x or 'food' in x else 0)
    df['is_grocery'] = df['facility_type'].apply(lambda x: 1 if 'grocery' in x or 'store' in x else 0)
    df['is_mobile'] = df['facility_type'].apply(lambda x: 1 if 'mobile' in x or 'truck' in x else 0)
    
    # Risk Category (Risk 1 = High, Risk 2 = Medium, Risk 3 = Low)
    df['risk_level'] = df['risk'].astype(str).apply(
        lambda x: 3 if 'Risk 1' in x else (2 if 'Risk 2' in x else (1 if 'Risk 3' in x else 0))
    )
    
    # Date parsing
    df['inspection_date'] = pd.to_datetime(df['inspection_date'])
    
    # Calculate historical features grouped by License #
    df = df.sort_values(by=['license_', 'inspection_date'])
    
    # 1. Days since last inspection
    df['prev_inspection_date'] = df.groupby('license_')['inspection_date'].shift(1)
    df['days_since_last_inspection'] = (df['inspection_date'] - df['prev_inspection_date']).dt.days
    # Fill NA (first-time inspections) with a high penalty value (e.g. 730 days / 2 years)
    df['days_since_last_inspection'] = df['days_since_last_inspection'].fillna(730)
    
    # 2. Cumulative historical failures
    df['historical_failures'] = df.groupby('license_')['target_failure'].cumsum().shift(1).fillna(0)
    
    # Select only the features we need
    features = ['is_restaurant', 'is_grocery', 'is_mobile', 'risk_level', 
                'days_since_last_inspection', 'historical_failures']
    
    X = df[features]
    y = df['target_failure']
    
    return X, y

X, y = engineer_features(raw_df)
print("Feature shape:", X.shape)
print("Failure distribution:\n", y.value_counts())

### 3. Compare Scikit-Learn Models
Train both Random Forest and XGBoost to see which handles the non-linear risks better.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, rf_preds))

# 2. XGBoost
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

print("\n=== XGBoost ===")
print(classification_report(y_test, xgb_preds))

### 4. Serialize the Winning Model
Save the highest performing model (likely XGBoost) for backend production use.

In [ ]:
# Export the XGBoost model as it typically yields better recall for rare events like failures
with open('global_baseline_xgb.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

print("Successfully saved global_baseline_xgb.pkl")
print("Feature mappings needed for API integration:", list(X.columns))